# DC1000 Dataset — Exploration

Explore the DC1000 caries segmentation dataset: image dimensions, mask properties,
caries coverage statistics, and sample visualizations.

In [ ]:
import glob
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Dataset structure and file counts

In [ ]:
DC1000 = "../data/DC1000"

for split in ['train', 'valid', 'test']:
    imgs = sorted(glob.glob(os.path.join(DC1000, split, 'images', '*.png')))
    masks = sorted(glob.glob(os.path.join(DC1000, split, 'masks', '*.png')))
    print(f"{split:6s}: {len(imgs)} images, {len(masks)} masks")
    assert len(imgs) == len(masks), f"Mismatch in {split}!"

## 2. Image dimensions

In [ ]:
# Check all image dimensions across all splits
all_shapes = set()
for split in ['train', 'valid', 'test']:
    imgs = sorted(glob.glob(os.path.join(DC1000, split, 'images', '*.png')))
    for p in imgs[:10]:  # spot-check 10 per split
        img = cv2.imread(p)
        all_shapes.add(img.shape)

print("Unique image shapes found:")
for s in all_shapes:
    print(f"  {s} (HxWxC)")

# Also check mask shapes
mask_shapes = set()
for split in ['train', 'valid', 'test']:
    masks = sorted(glob.glob(os.path.join(DC1000, split, 'masks', '*.png')))
    for p in masks[:10]:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        mask_shapes.add(m.shape)

print("\nUnique mask shapes found:")
for s in mask_shapes:
    print(f"  {s} (HxW)")

## 3. Mask value analysis

In [ ]:
# Check unique mask values across dataset
all_unique_vals = set()
for split in ['train', 'valid', 'test']:
    masks = sorted(glob.glob(os.path.join(DC1000, split, 'masks', '*.png')))
    for p in masks[:20]:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        all_unique_vals.update(set(np.unique(m)))

print(f"Unique mask pixel values: {sorted(all_unique_vals)}")
print("→ Binary masks: 0 = background, 255 = caries")

## 4. Caries coverage statistics

What percentage of each image is covered by caries? This tells us about class imbalance.

In [ ]:
coverages = {'train': [], 'valid': [], 'test': []}
zero_mask_counts = {'train': 0, 'valid': 0, 'test': 0}

for split in ['train', 'valid', 'test']:
    masks = sorted(glob.glob(os.path.join(DC1000, split, 'masks', '*.png')))
    for p in masks:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        caries_pixels = (m > 127).sum()
        total_pixels = m.shape[0] * m.shape[1]
        coverage = caries_pixels / total_pixels * 100
        coverages[split].append(coverage)
        if caries_pixels == 0:
            zero_mask_counts[split] += 1

for split in ['train', 'valid', 'test']:
    c = coverages[split]
    print(f"{split:6s}: mean={np.mean(c):.3f}%, median={np.median(c):.3f}%, "
          f"max={np.max(c):.3f}%, min={np.min(c):.3f}%, "
          f"zero-mask images={zero_mask_counts[split]}")

In [ ]:
# Coverage distribution plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, split in zip(axes, ['train', 'valid', 'test']):
    c = coverages[split]
    ax.hist(c, bins=30, color='steelblue', edgecolor='black')
    ax.set_title(f'{split} — Caries Coverage Distribution')
    ax.set_xlabel('Coverage (%)')
    ax.set_ylabel('Image count')
    ax.axvline(np.mean(c), color='red', linestyle='--', label=f'mean={np.mean(c):.3f}%')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Sample panoramic images with their masks

In [ ]:
# Show 4 samples: original image + mask overlay
train_imgs = sorted(glob.glob(os.path.join(DC1000, 'train', 'images', '*.png')))
train_masks = sorted(glob.glob(os.path.join(DC1000, 'train', 'masks', '*.png')))

# Pick images with varying caries coverage
rng = np.random.RandomState(42)
indices = rng.choice(len(train_imgs), size=4, replace=False)

fig, axes = plt.subplots(4, 3, figsize=(24, 20))

for row, idx in enumerate(indices):
    img = cv2.imread(train_imgs[idx])
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(train_masks[idx], cv2.IMREAD_GRAYSCALE)
    
    # Create overlay
    overlay = img_rgb.copy()
    mask_bool = mask > 127
    overlay[mask_bool] = [255, 0, 0]  # Red for caries
    blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)
    
    coverage = mask_bool.sum() / mask.size * 100
    name = os.path.basename(train_imgs[idx])
    
    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_title(f'{name} — Original')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(mask, cmap='gray')
    axes[row, 1].set_title(f'Mask — {coverage:.3f}% caries')
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title('Overlay (red = caries)')
    axes[row, 2].axis('off')

plt.suptitle('DC1000 — Sample Panoramic Images with Caries Masks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Caries region size analysis

How large are individual caries regions? This is relevant for understanding what Stage 2 will see per crop.

In [ ]:
# Find connected components in masks to analyze individual caries regions
region_areas = []
regions_per_image = []

for split in ['train', 'valid', 'test']:
    masks = sorted(glob.glob(os.path.join(DC1000, split, 'masks', '*.png')))
    for p in masks:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        binary = (m > 127).astype(np.uint8)
        n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)
        # Skip label 0 (background)
        regions_per_image.append(n_labels - 1)
        for i in range(1, n_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            region_areas.append(area)

print(f"Total caries regions found: {len(region_areas)}")
print(f"Regions per image: min={min(regions_per_image)}, max={max(regions_per_image)}, "
      f"mean={np.mean(regions_per_image):.1f}, median={np.median(regions_per_image):.0f}")
if region_areas:
    print(f"Region area (px): min={min(region_areas)}, max={max(region_areas)}, "
          f"mean={np.mean(region_areas):.0f}, median={np.median(region_areas):.0f}")

In [ ]:
if region_areas:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(region_areas, bins=50, color='coral', edgecolor='black')
    axes[0].set_title('Individual Caries Region Area Distribution')
    axes[0].set_xlabel('Area (pixels)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(np.median(region_areas), color='red', linestyle='--',
                    label=f'median={np.median(region_areas):.0f}')
    axes[0].legend()
    
    axes[1].hist(regions_per_image, bins=range(0, max(regions_per_image) + 2),
                 color='mediumpurple', edgecolor='black')
    axes[1].set_title('Caries Regions Per Image')
    axes[1].set_xlabel('Number of regions')
    axes[1].set_ylabel('Image count')
    
    plt.tight_layout()
    plt.show()

## 7. Size comparison: Tufts vs DC1000

The YOLO model trained on Tufts (840×1615) must transfer to DC1000 (1435×2943).
Both are panoramic radiographs but at different resolutions.

In [ ]:
print("Resolution comparison:")
print(f"  Tufts dental:  840 × 1615  (aspect ratio: {1615/840:.2f})")
print(f"  DC1000:       1435 × 2943  (aspect ratio: {2943/1435:.2f})")
print(f"  Scale factor:  {1435/840:.2f}× height, {2943/1615:.2f}× width")
print(f"\nBoth are ~1.9:1 aspect ratio — good for transfer.")
print(f"YOLO normalizes to 640×640 internally, so absolute resolution difference is handled.")

## 8. Summary

In [ ]:
total_imgs = sum(len(glob.glob(os.path.join(DC1000, s, 'images', '*.png')))
                 for s in ['train', 'valid', 'test'])

print("=" * 50)
print("DC1000 DATASET SUMMARY")
print("=" * 50)
print(f"Total images:        {total_imgs}")
print(f"  Train:             394")
print(f"  Valid:             99")
print(f"  Test:              100")
print(f"Image dimensions:    1435 × 2943 (H × W) — all consistent")
print(f"Aspect ratio:        ~2.05:1")
print(f"Mask type:           Binary (0/255)")
print(f"File format:         .png")
print(f"Mean caries coverage: {np.mean(coverages['train']):.3f}% (train)")
print(f"Zero-mask images:    train={zero_mask_counts['train']}, "
      f"valid={zero_mask_counts['valid']}, test={zero_mask_counts['test']}")
if region_areas:
    print(f"Caries regions/img:  {np.mean(regions_per_image):.1f} avg")
    print(f"Median region size:  {np.median(region_areas):.0f} px")
print("=" * 50)